Exploratory analysis (chapter Analysis), descriptive only. Same answers, weights and bootstrap draws as `scoring.ipynb`.
Writes `results/analysis.json` (flat key → value, merged into `numbers.tex` by `figures.ipynb`).
Already in `results.json`: yes rate per bin, BA per ecoregion / stratum / season. Failure-mode categories: `rq3_failures.ipynb`.

In [15]:
import json

import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.linear_model import LogisticRegression

from thesis.config import REPO_ROOT, load_config
from thesis.eval.scorer import item_weights, percentile_interval, predictions, score

out_dir = REPO_ROOT / "data" / "dataset_v2"
results_dir = REPO_ROOT / "results"
scoring = load_config("scoring")


def bin_key(b):
    return f"m{-b}" if b < 0 else f"p{b}"


excluded = {entry["source_id"] for entry in scoring["excluded_sources"]}
sources = pd.read_parquet(out_dir / "selection" / "sources.parquet")[
    ["source_id", "box_id", "ecoregion", "building_stratum"]
]
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")[
    ["pin_id", "nearest_polygon_provenance", "nearest_edge_bearing_radians"]
]
answers = (
    pd.read_parquet(out_dir / "main_run" / "answers.parquet").merge(sources, on="source_id").merge(pins, on="pin_id")
)
answers["inside"] = answers["bin_id"] < 0
primary = answers.loc[(answers["purpose"] == "primary") & ~answers["source_id"].isin(excluded)]
verification = primary.loc[primary["task"] == "verification"]
distance = primary.loc[primary["task"] == "distance"]
MODELS = sorted(primary["model"].unique())

analysis = {}


def record(key, point, reps):
    low, high = percentile_interval(reps)
    analysis.update({key: float(point), f"{key}_ci_low": low, f"{key}_ci_high": high})

In [16]:
# designed factors: accuracy per bin for each ecoregion and each building stratum
for name, column in {"ecoregion": "ecoregion", "stratum": "building_stratum"}.items():
    point, reps = score(verification, by=["model", column, "bin_id"])
    for (m, level, b), value in point["accuracy"].items():
        record(f"verification_accuracy_{m}_{bin_key(b)}_{name}_{level}", value, reps[("accuracy", m, level, b)])
len(analysis)

936

In [17]:
# observed factors: provenance of the nearest footprint, bearing pin -> nearest edge (0 = east, counterclockwise)
from thesis.run import _provenance_category


def bearing_class(radians):
    return np.array(["east", "north", "west", "south"])[((np.degrees(radians) + 45) % 360 // 90).astype(int)]


for frame in (verification, distance):
    frame["provenance"] = (
        frame["nearest_polygon_provenance"].map(_provenance_category).str.lower().str.replace(" ", "_")
    )
    frame["bearing"] = bearing_class(frame["nearest_edge_bearing_radians"].to_numpy())

for name in ("provenance", "bearing"):
    point, reps = score(verification, by=["model", name])
    for (m, level), value in point["balanced_accuracy"].items():
        record(f"verification_ba_{m}_{name}_{level}", value, reps[("balanced_accuracy", m, level)])
    point, reps = score(distance, by=["model", name], sign_from_distance=True)
    for (m, level), value in point["balanced_accuracy"].items():
        record(f"distance_sign_ba_{m}_{name}_{level}", value, reps[("balanced_accuracy", m, level)])
(
    verification.groupby(["model", "provenance"]).size().unstack(0),
    verification.groupby(["model", "bearing"]).size().unstack(0),
)

(model                   anthropic  google  openai
 provenance                                       
 esri_community_maps           544     544     544
 microsoft_ml_buildings       9820    9820    9820
 openstreetmap                3841    3841    3841,
 model    anthropic  google  openai
 bearing                           
 east          3505    3505    3505
 north         3699    3699    3699
 south         3598    3598    3598
 west          3403    3403    3403)

In [18]:
# task 1 vs sign of task 2 on the same pins (unweighted counts, invalid answers as their own class)
verdict = verification[["model", "pin_id", "parsed_answer"]].rename(columns={"parsed_answer": "task1"})
sign = pd.to_numeric(distance["parsed_answer"], errors="coerce")
verdict = verdict.merge(
    distance[["model", "pin_id"]].assign(task2=np.select([sign <= 0, sign > 0], ["inside", "outside"], "invalid")),
    on=["model", "pin_id"],
)
verdict["task1"] = verdict["task1"].where(verdict["task1"].isin(["yes", "no"]), "invalid")
for m, group in verdict.groupby("model"):
    agree = ((group["task1"] == "yes") & (group["task2"] == "inside")) | (
        (group["task1"] == "no") & (group["task2"] == "outside")
    )
    analysis[f"task_agreement_{m}"] = float(agree.mean())
    analysis[f"task_yes_outside_{m}"] = int(((group["task1"] == "yes") & (group["task2"] == "outside")).sum())
    analysis[f"task_no_inside_{m}"] = int(((group["task1"] == "no") & (group["task2"] == "inside")).sum())
pd.crosstab([verdict["model"], verdict["task1"]], verdict["task2"])

task2              inside  invalid  outside
model     task1                            
anthropic invalid      16        1      155
          no          344        4    11469
          yes         736        1     1479
google    no          399        0     9712
          yes        3450        0      644
openai    no          513        0     8337
          yes        3863        0     1492

In [19]:
# psychometric fit (exploratory): weighted logistic regression of outside correctness on ln(exact distance),
# design weights, invalid = wrong; d_tau = distance where the fit reaches tau (point estimate only)
outside = predictions(verification.loc[~verification["inside"]], scoring["invalid_policy"])
outside = outside.assign(weight=item_weights(outside, scoring["estimand"]), correct=~outside["predicted_inside"])
for m, group in outside.groupby("model"):
    fit = LogisticRegression(C=np.inf).fit(
        np.log(group[["exact_signed_distance_m"]]), group["correct"], sample_weight=group["weight"]
    )
    intercept, slope = fit.intercept_[0], fit.coef_[0, 0]
    analysis.update(
        {
            f"psychometric_intercept_{m}": float(intercept),
            f"psychometric_slope_{m}": float(slope),
            f"psychometric_d_tau_{m}": float(np.exp((logit(scoring["tau"]) - intercept) / slope)),
        }
    )
{k: round(v, 2) for k, v in analysis.items() if k.startswith("psychometric_d_tau")}

{'psychometric_d_tau_anthropic': 15.89,
 'psychometric_d_tau_google': 10.57,
 'psychometric_d_tau_openai': 17.29}

In [20]:
# edge-marked pins (exploratory): VLM task 1 answer and point-in-polygon on Overture against the visible label.
# visible signed distance = d - e (e = visible roof edge outside the footprint edge, from the click); unweighted, box bootstrap
import geopandas as gpd
from shapely import from_wkb
from shapely.geometry import Point
from shapely.ops import nearest_points

from thesis.data.eligibility import context_for_scene
from thesis.run import _geometry_to_crs, _valid_context_footprints

marks = pd.read_json(out_dir / "gold" / "edge_marks_phillip.jsonl", lines=True).query("~skipped")
scenes = pd.read_parquet(out_dir / "selection" / "sources.parquet")[
    ["source_id", "naip_source_crs", "scene_geometry_wkb"]
]
edge_pins = (
    marks[["pin_id", "click_x", "click_y"]]
    .merge(
        pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")[
            ["pin_id", "source_id", "tip_x", "tip_y", "exact_signed_distance_m"]
        ],
        on="pin_id",
    )
    .merge(scenes, on="source_id")
    .loc[lambda f: ~f["source_id"].isin(excluded)]
)
footprints = _valid_context_footprints(gpd.read_parquet(REPO_ROOT / load_config("overture")["out"]))


def displacement(row):
    scene = _geometry_to_crs(from_wkb(row["scene_geometry_wkb"]), "EPSG:5070", "EPSG:4326")
    edge = context_for_scene(footprints, scene).to_crs(row["naip_source_crs"]).union_all().boundary
    pin = np.array([row["tip_x"], row["tip_y"]])
    r = np.array(nearest_points(edge, Point(pin))[0].coords[0])
    e = (np.array([row["click_x"], row["click_y"]]) - r) @ ((r - pin) / np.linalg.norm(r - pin))
    return -e if row["exact_signed_distance_m"] > 0 else e


edge_pins["visible_inside"] = edge_pins["exact_signed_distance_m"] - edge_pins.apply(displacement, axis=1) < 0
edge_answers = answers.loc[(answers["purpose"] == "edge") & (answers["task"] == "verification")]
overture = edge_answers.loc[edge_answers["model"] == MODELS[0]].assign(
    model="overture",
    parsed_answer=np.where(
        edge_answers.loc[edge_answers["model"] == MODELS[0], "exact_signed_distance_m"] < 0, "yes", "no"
    ),
)
edge_items = (
    pd.concat([edge_answers, overture])
    .merge(edge_pins[["pin_id", "visible_inside"]], on="pin_id")
    .assign(inside=lambda f: f["visible_inside"])
)
assert (edge_items.groupby("model").size() == len(edge_pins)).all()

analysis["edge_n_pins"] = len(edge_pins)
analysis["edge_visible_flips"] = int((edge_pins["visible_inside"] != (edge_pins["exact_signed_distance_m"] < 0)).sum())
point, reps = score(edge_items, by=["model"], estimand="pooled")
for m in [*MODELS, "overture"]:
    record(f"edge_visible_accuracy_{m}", point.loc[m, "accuracy"], reps[("accuracy", m)])
    record(f"edge_visible_ba_{m}", point.loc[m, "balanced_accuracy"], reps[("balanced_accuracy", m)])
point.round(3)

,balanced_accuracy,accuracy,precision
model,,,
anthropic,0.557,0.622,0.451
google,0.642,0.606,0.464
openai,0.564,0.487,0.392
overture,0.726,0.710,0.564


In [22]:
(results_dir / "analysis.json").write_text(json.dumps(analysis, indent=1))
len(analysis)

1106